# Layer Normalization — Implementation / 구현

**Paper**: Ba, J. L., Kiros, J. R., & Hinton, G. E. (2016). *Layer Normalization*. arXiv:1607.06450.

This notebook implements Layer Normalization from scratch and empirically explores its properties. We compare LN against BN, IN, GN on a CNN tensor; demonstrate LN's batch-size independence; verify invariance properties; and apply LN to an LSTM.

이 노트북은 LayerNorm을 바닥부터 구현하고, BN/IN/GN과 비교하며, 배치-크기 독립성과 불변성을 실험적으로 검증합니다.

## Contents / 목차
1. **LayerNorm from scratch** — PyTorch에서 바닥부터 구현, nn.LayerNorm과 일치 검증
2. **BN vs LN vs IN vs GN on a CNN tensor** — 4가지 정규화의 pooling 축 시각화 및 통계 수 확인
3. **Batch-size independence** — LN은 batch=1에서도 작동, BN은 실패
4. **Invariance properties** — Table 1 검증: per-sample rescaling & shift 불변성
5. **Applying LN inside an LSTM** — §4의 LN-LSTM 구현, LN이 gradient norm에 미치는 영향
6. **RMSNorm — the modern simplification** — LN에서 mean subtraction 생략한 LLaMA-style norm
7. **Summary Table** — 원 논문 vs 현대 실용

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## Part 1: LayerNorm from Scratch / 바닥부터 구현

논문 식 (3)을 그대로 옮깁니다:
$$\mu = \frac{1}{H}\sum_i a_i, \quad \sigma^2 = \frac{1}{H}\sum_i (a_i - \mu)^2, \quad y_i = g_i \frac{a_i - \mu}{\sqrt{\sigma^2 + \epsilon}} + b_i$$

We implement the paper's Eq. (3) literally, then cross-check against `nn.LayerNorm`.

In [ ]:
class MyLayerNorm(nn.Module):
    """Layer Normalization from scratch (Ba et al. 2016).

    Normalizes over the last `normalized_shape` dimensions of the input,
    independently per sample — no batch coupling.
    """
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.eps = eps
        self.g = nn.Parameter(torch.ones(normalized_shape))
        self.b = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        # Axes to reduce = last len(normalized_shape) dims.
        dims = tuple(range(-len(self.normalized_shape), 0))
        mu = x.mean(dim=dims, keepdim=True)
        var = x.var(dim=dims, keepdim=True, unbiased=False)
        x_hat = (x - mu) / torch.sqrt(var + self.eps)
        return x_hat * self.g + self.b


# Sanity check: match nn.LayerNorm on a random tensor.
x = torch.randn(4, 8, 16)  # (N, T, H) — classic Transformer-like shape
my_ln = MyLayerNorm(16)
ref_ln = nn.LayerNorm(16)
# Copy params so they match.
with torch.no_grad():
    ref_ln.weight.copy_(my_ln.g)
    ref_ln.bias.copy_(my_ln.b)

max_diff = (my_ln(x) - ref_ln(x)).abs().max().item()
print(f'Max diff vs nn.LayerNorm: {max_diff:.2e}  →  {"MATCH ✓" if max_diff < 1e-5 else "MISMATCH"}')

## Part 2: BN vs LN vs IN vs GN on a CNN Tensor / 4가지 정규화 비교

briefing Q2의 RGB 4배치 예시를 직접 확인합니다. Shape $(N, C, H, W) = (4, 3, 8, 8)$에서 각 정규화가 계산하는 통계의 수와 pooling 대상을 검증.

We verify the statistic counts from briefing Q2: BN→3, LN→4, IN→12, GN(G=3)→12.

In [ ]:
def norm_stats(x, pool_dims):
    """Return mean and variance of x pooled over `pool_dims`.

    Args:
        x: tensor.
        pool_dims: tuple of dims to average over.
    Returns:
        (mu, var) — same shape as x with pool_dims squeezed.
    """
    mu = x.mean(dim=pool_dims, keepdim=True)
    var = x.var(dim=pool_dims, keepdim=True, unbiased=False)
    return mu, var


# Build a CNN tensor
N, C, H, W = 4, 3, 8, 8
x_cnn = torch.randn(N, C, H, W)
print(f'Input shape: {tuple(x_cnn.shape)}\n')

# BN: pool over (N, H, W) → C statistics
mu_bn, var_bn = norm_stats(x_cnn, pool_dims=(0, 2, 3))
print(f'BN:  mu shape = {tuple(mu_bn.squeeze().shape) or "scalar"} → {mu_bn.numel()} stats  (should be C={C})')

# LN: pool over (C, H, W) → N statistics
mu_ln, var_ln = norm_stats(x_cnn, pool_dims=(1, 2, 3))
print(f'LN:  mu shape = {tuple(mu_ln.squeeze().shape) or "scalar"} → {mu_ln.numel()} stats  (should be N={N})')

# IN: pool over (H, W) → N*C statistics
mu_in, var_in = norm_stats(x_cnn, pool_dims=(2, 3))
print(f'IN:  mu shape = {tuple(mu_in.squeeze().shape) or "scalar"} → {mu_in.numel()} stats  (should be N*C={N*C})')

# GN with G=3 groups: reshape to (N, G, C/G, H, W), pool over (C/G, H, W)
G = 3
x_grouped = x_cnn.view(N, G, C // G, H, W)
mu_gn, var_gn = norm_stats(x_grouped, pool_dims=(2, 3, 4))
print(f'GN:  mu shape = {tuple(mu_gn.squeeze().shape) or "scalar"} → {mu_gn.numel()} stats  (should be N*G={N*G})')

In [ ]:
# Visualize which tensor cells share statistics under each scheme.
# We'll color a (N=4, C=3) slice (averaging over H,W for display).

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['BN: group by channel',
          'LN: group by sample',
          'IN: group by (sample, channel)',
          'GN (G=3): same as IN here']

# Build color labels: same number = same stats.
grid_bn = np.tile(np.arange(C), (N, 1))        # column-wise
grid_ln = np.tile(np.arange(N).reshape(-1, 1), (1, C))  # row-wise
grid_in = np.arange(N * C).reshape(N, C)
grid_gn = grid_in  # since G=C, each channel is its own group

for ax, title, grid in zip(axes, titles, [grid_bn, grid_ln, grid_in, grid_gn]):
    ax.imshow(grid, cmap='tab20', aspect='auto')
    ax.set_xticks(range(C)); ax.set_xticklabels(['R', 'G', 'B'])
    ax.set_yticks(range(N)); ax.set_yticklabels([f's{i+1}' for i in range(N)])
    ax.set_title(title)
    for i in range(N):
        for j in range(C):
            ax.text(j, i, str(grid[i, j]), ha='center', va='center', fontsize=9)

axes[0].set_ylabel('Batch sample')
plt.suptitle('Same color/number = cells sharing (μ, σ). N=4 samples × C=3 (RGB).')
plt.tight_layout(); plt.show()

**관찰 / Observation**: BN은 열(채널)별로 묶고, LN은 행(샘플)별로 묶습니다. IN/GN(G=C)은 각 셀이 독립. 이 그림이 "같은 행렬의 평균을 다른 축으로 구한다"는 LN의 본질을 시각적으로 보여줍니다.

BN groups by column (channel); LN by row (sample); IN/GN(G=C) treats each cell alone. This visually captures "averaging the same tensor along different axes."

## Part 3: Batch-size Independence / 배치 크기 독립성

LN의 핵심 실용적 장점: **배치 크기 1에서도 분산이 0이 아님**. 같은 샘플을 batch=1, 4, 16으로 넣어도 LN 출력은 동일하지만, BN은 batch=1에서 무너집니다.

We verify: a sample processed under different batch sizes gives identical LN output (as long as batch contains it), but different BN output. BN fails at batch=1.

In [ ]:
torch.manual_seed(1)
sample = torch.randn(1, 16)  # one sample, 16 features
filler = torch.randn(15, 16)  # noise to pad batches

ln = nn.LayerNorm(16)
bn = nn.BatchNorm1d(16)
bn.train()  # training mode uses batch stats

print('=== LayerNorm ===')
for B in [1, 4, 16]:
    batch = torch.cat([sample, filler[:B-1]], dim=0) if B > 1 else sample
    out = ln(batch)
    print(f'batch={B:>2}  | sample output[:4] = {out[0, :4].detach().numpy().round(3)}')

print('\n=== BatchNorm (training mode) ===')
for B in [1, 4, 16]:
    batch = torch.cat([sample, filler[:B-1]], dim=0) if B > 1 else sample
    # Re-init BN to avoid running-stats pollution.
    bn_fresh = nn.BatchNorm1d(16); bn_fresh.train()
    try:
        out = bn_fresh(batch)
        print(f'batch={B:>2}  | sample output[:4] = {out[0, :4].detach().numpy().round(3)}')
    except Exception as e:
        print(f'batch={B:>2}  | FAILED: {type(e).__name__}: {str(e)[:80]}')

**관찰 / Observation**: LN 출력은 배치 크기와 상관없이 **완전히 동일**합니다 (각 샘플이 자기 통계로 정규화되니까). BN은 batch=1에서 단일 샘플로 분산 계산이 불가능해 오류를 내거나 의미 없는 출력을 냅니다. 이것이 LLM inference (보통 batch=1), detection/segmentation의 작은 배치, 그리고 online learning에서 LN이 사실상 유일한 선택인 이유입니다.

LN outputs are **identical** regardless of batch size — each sample uses its own stats. BN cannot compute meaningful statistics at batch=1. This is why LN is essentially the only choice for LLM inference (usually batch=1), small-batch detection/segmentation, and online learning.

## Part 4: Invariance Properties / 불변성 검증 (Table 1)

논문 Table 1의 핵심 주장 두 가지를 검증합니다:
- **Per-sample rescaling invariance**: 같은 샘플을 $\alpha \mathbf{x}$로 스케일해도 LN 출력 불변.
- **Per-sample shift invariance**: 같은 샘플에 상수 $c$를 더해도 LN 출력 불변.

We verify the two claims from Table 1: LN output is invariant to both per-sample rescaling and shifting.

In [ ]:
ln = nn.LayerNorm(32)
bn = nn.BatchNorm1d(32); bn.train()

x0 = torch.randn(4, 32)

# Apply per-sample rescaling: sample n → alpha_n * sample_n
alphas = torch.tensor([0.5, 2.0, 5.0, 0.1]).view(-1, 1)
x_rescaled = x0 * alphas

# Apply per-sample shift: sample n → sample_n + c_n
shifts = torch.tensor([-3.0, 1.5, 0.0, 7.0]).view(-1, 1)
x_shifted = x0 + shifts

def max_diff(a, b):
    return (a - b).abs().max().item()

print('=== Per-sample rescale (x → α·x per sample) ===')
print(f'LN output diff: {max_diff(ln(x0), ln(x_rescaled)):.2e}   (paper claims: 0 ✓ if tiny)')
print(f'BN output diff: {max_diff(bn(x0), bn(x_rescaled)):.2e}   (paper claims: nonzero ✓ if large)')

print('\n=== Per-sample shift (x → x + c per sample) ===')
print(f'LN output diff: {max_diff(ln(x0), ln(x_shifted)):.2e}   (paper claims: 0 ✓ if tiny)')
print(f'BN output diff: {max_diff(bn(x0), bn(x_shifted)):.2e}   (paper claims: nonzero ✓ if large)')

**관찰 / Observation**: LN은 per-sample rescaling과 shifting 모두에 **수치 오차 수준으로** 불변입니다 (Table 1의 주장 확인). BN은 두 경우 모두 출력이 크게 달라집니다. 이 성질 덕분에 LN은 시퀀스마다 스케일이 다른 RNN 입력이나, 전반적 밝기가 다른 이미지 토큰에도 강건합니다.

LN is invariant (to numerical precision) to per-sample rescaling and shifting — exactly Table 1's claim. BN changes substantially. This is why LN handles variable-scale RNN inputs and brightness-varying tokens without trouble.

## Part 5: LN inside an LSTM / LSTM에 LN 적용

논문 §4 식 (20)–(21)에 따라 LN-LSTM을 구현합니다. LN은 gate pre-activation에 삽입되며, 모든 time step이 **같은** $\mathbf{g}, \mathbf{b}$를 공유합니다.

We implement LN-LSTM per §4 Eqs. (20)–(21). LN is inserted at gate pre-activations; $\mathbf{g}, \mathbf{b}$ are shared across time steps.

In [ ]:
class LNLSTMCell(nn.Module):
    """LSTM cell with Layer Normalization on the pre-activation.

    Follows paper §4 Eqs. (20)-(21). LN parameters are shared across
    all time steps (unlike BN, which would need per-step tracking).
    """
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        # Combined weight for all 4 gates: i, f, o, g
        self.W_ih = nn.Linear(input_dim, 4 * hidden_dim, bias=False)
        self.W_hh = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)
        # Single LN on the 4H-dim pre-activation (shared across time).
        self.ln = nn.LayerNorm(4 * hidden_dim)
        # Separate LN on cell state (also in the paper).
        self.ln_c = nn.LayerNorm(hidden_dim)

    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = self.ln(self.W_ih(x_t) + self.W_hh(h_prev))
        i, f, g, o = gates.chunk(4, dim=-1)
        i = torch.sigmoid(i); f = torch.sigmoid(f)
        o = torch.sigmoid(o); g = torch.tanh(g)
        c_t = f * c_prev + i * g
        h_t = o * torch.tanh(self.ln_c(c_t))
        return h_t, c_t


class PlainLSTMCell(nn.Module):
    """Vanilla LSTM cell — same shape, no LN, for comparison."""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.W_ih = nn.Linear(input_dim, 4 * hidden_dim)
        self.W_hh = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)

    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = self.W_ih(x_t) + self.W_hh(h_prev)
        i, f, g, o = gates.chunk(4, dim=-1)
        i = torch.sigmoid(i); f = torch.sigmoid(f)
        o = torch.sigmoid(o); g = torch.tanh(g)
        c_t = f * c_prev + i * g
        h_t = o * torch.tanh(c_t)
        return h_t, c_t


def run_sequence(cell, x_seq, H):
    """Run a cell over a sequence, returning hidden states at each step."""
    N = x_seq.shape[1]
    h = torch.zeros(N, H); c = torch.zeros(N, H)
    outputs = []
    for t in range(x_seq.shape[0]):
        h, c = cell(x_seq[t], (h, c))
        outputs.append(h)
    return torch.stack(outputs)


# Toy example: generate a 50-step sequence with growing scale (stressing the LSTM)
T, N, D, H = 50, 16, 32, 64
torch.manual_seed(2)
scale = torch.linspace(1.0, 10.0, T).view(T, 1, 1)  # scale grows over time
x_seq = torch.randn(T, N, D) * scale

torch.manual_seed(3)
plain = PlainLSTMCell(D, H)
torch.manual_seed(3)
ln_lstm = LNLSTMCell(D, H)

h_plain = run_sequence(plain, x_seq, H)
h_ln = run_sequence(ln_lstm, x_seq, H)

# Track activation magnitude per time step
mag_plain = h_plain.detach().pow(2).mean(dim=(1, 2)).sqrt()
mag_ln    = h_ln.detach().pow(2).mean(dim=(1, 2)).sqrt()

plt.figure()
plt.plot(mag_plain.numpy(), 'o-', label='Plain LSTM')
plt.plot(mag_ln.numpy(),    's-', label='LN-LSTM')
plt.xlabel('Time step'); plt.ylabel('Mean hidden state RMS')
plt.title('Hidden state magnitude on a scale-growing sequence')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f'Plain LSTM final-step RMS: {mag_plain[-1]:.4f}')
print(f'LN-LSTM    final-step RMS: {mag_ln[-1]:.4f}')

**관찰 / Observation**: Plain LSTM은 입력 스케일이 증가할 때 hidden state 크기가 불안정하게 변합니다. LN-LSTM은 각 time step마다 pre-activation을 자기 통계로 정규화하므로 **hidden state 크기가 안정적으로 유지**됩니다. 이것이 논문 §6의 RNN 실험에서 LN이 더 빠른 수렴과 더 나은 최종 성능을 얻은 이유입니다.

Plain LSTM's hidden state magnitude drifts with input scale; LN-LSTM stays stable because each time step re-normalizes its own pre-activation. This is why §6 experiments show LN gives both faster convergence and better final accuracy in RNNs.

## Part 6: RMSNorm — The Modern Simplification / 현대적 단순화

2019년 Zhang & Sennrich는 LN에서 mean subtraction이 크게 기여하지 않는다는 실증적 관찰 위에 **RMSNorm**을 제안:
$$\text{RMSNorm}(x)_i = \frac{x_i}{\sqrt{\frac{1}{H}\sum_j x_j^2 + \epsilon}} \cdot g_i$$

더 빠르고 파라미터가 적으며, LLaMA 계열이 채택했습니다. 직접 구현하고 LN과 비교합니다.

RMSNorm (Zhang & Sennrich 2019) drops LN's mean subtraction — faster, fewer ops, adopted by LLaMA.

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (Zhang & Sennrich, 2019).

    Drops mean subtraction from LN. Only the scale (RMS) is removed.
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.g = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).sqrt()
        return x / (rms + self.eps) * self.g


# Compare LN vs RMSNorm on a random input
x_test = torch.randn(4, 128) * 3.0 + 1.5  # nonzero mean, scale 3
ln = nn.LayerNorm(128)
rms = RMSNorm(128)

y_ln = ln(x_test)
y_rms = rms(x_test)

print(f'Input mean per sample    : {x_test.mean(dim=-1).detach().numpy().round(3)}')
print(f'Input std per sample     : {x_test.std(dim=-1).detach().numpy().round(3)}')
print(f'LN output mean per sample: {y_ln.mean(dim=-1).detach().numpy().round(3)}  (should be ~0)')
print(f'RMSNorm output mean      : {y_rms.mean(dim=-1).detach().numpy().round(3)}  (NOT ~0 — mean not subtracted)')
print(f'LN output std per sample : {y_ln.std(dim=-1).detach().numpy().round(3)}')
print(f'RMSNorm output std       : {y_rms.std(dim=-1).detach().numpy().round(3)}')

# Compute operation count (ops per forward pass, roughly)
H = 128
ops_ln  = 3 * H  # sum for mean, sum for var, normalize+affine
ops_rms = 2 * H  # sum of squares, normalize+affine
print(f'\nRough ops per token: LN ~{ops_ln}, RMSNorm ~{ops_rms}  → RMSNorm is ~{ops_ln/ops_rms:.2f}× faster.')

**관찰 / Observation**: RMSNorm은 mean을 빼지 않으므로 출력의 평균이 0이 아닙니다 (그러나 scale은 여전히 정규화). 파라미터 수도 절반 ($g$만), 연산량도 약 1.5배 적습니다. LLaMA, Qwen, Mistral 등 현대 오픈 LLM이 이를 채택하는 이유입니다.

RMSNorm leaves the mean intact (scale still normalized). Half the parameters, ~1.5× fewer ops — why LLaMA, Qwen, Mistral adopt it.

## Summary / 요약

| Concept / 개념 | This Paper (2016) / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Normalization axis | Within-sample (over H hidden units) | Same — still the LN definition in every LLM |
| Main target | RNN / fully-connected | Transformer (universal), plus ViT, diffusion |
| Train/infer parity | Identical computation | Same |
| Batch-size independence | Core feature | Foundation of LLM inference at batch=1 |
| Affine params | Per-unit $g_i, b_i$ | Same; RMSNorm drops $b_i$ (and mean subtraction) |
| CNN applicability | Underperforms BN (§6.7) | Still true; CNNs use BN or GroupNorm |
| Placement | After pre-activation, before nonlinearity | In Transformer: pre-norm (before sub-layer) now standard |
| Simplifications | — | RMSNorm (LLaMA, 2019), ScaleNorm |
| Generalizations | — | GroupNorm (Wu-He 2018), InstanceNorm |

### Key insights reproduced / 재현된 핵심 인사이트
- **Part 1**: Paper Eq. (3)을 바닥부터 구현, nn.LayerNorm과 완전 일치 확인.
- **Part 2**: BN/LN/IN/GN이 같은 CNN tensor에서 서로 다른 축으로 pool함을 시각화 (briefing Q2 검증).
- **Part 3**: LN이 배치 크기와 완전 독립, BN은 batch=1에서 붕괴.
- **Part 4**: Table 1의 per-sample rescale/shift 불변성 수치 확인.
- **Part 5**: LN-LSTM이 입력 스케일 변화에서 hidden state를 안정시키는 효과 시연.
- **Part 6**: LN의 현대 계승자 RMSNorm 구현 및 비교.

이 여섯 가지가 "왜 LN이 Transformer와 LLM 시대의 기본 정규화가 되었는가"에 대한 정량적 답입니다.

Together these six experiments quantitatively answer why LN became the default normalization of the Transformer / LLM era.